# Image Manipulation With Pillow and NumPy

The first cell installs the lab's libraries with pip, Python's package manager: `%pip install` installs a package, with its dependencies, into the notebook's environment. Colab preinstalls them: pip answers `Requirement already satisfied` for each one, and installs nothing. On your own machine, in a conda environment, the equivalent is `conda install pillow numpy matplotlib`, in a terminal where the environment is activated.

The second cell downloads the lab's data: the LFW dataset (*Labeled Faces in the Wild*), 13,233 photos of faces of 250 × 250 pixels, stored in the `lfw` folder with one folder per person (180 MB, which the cell downloads then extracts: allow up to a minute), and the `bandana.jpg` photo of the last exercise.

The third cell imports the libraries and defines `side_by_side(images, title=None, subtitles=None)`, which displays images side by side: the solutions use it, and so can you.

In [ ]:
%pip install pillow numpy matplotlib

In [ ]:
!rm -rf sample_data
!wget -q https://ndownloader.figshare.com/files/5976018 -O lfw.tgz
!wget -q -O bandana.jpg https://github.com/shuuchuu/datasets/raw/refs/heads/main/rosie-the-riveter/rosie-the-riveter.jpg
!tar -xf lfw.tgz
!rm lfw.tgz

In [ ]:
import pathlib
from collections.abc import Sequence

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageFilter, ImageOps


def side_by_side(
  images: Sequence[Image.Image | np.ndarray],
  title: str | None = None,
  subtitles: Sequence[str] | None = None,
  dpi: int = 150,
) -> None:
  """Display images side by side, each under its subtitle.

  A single-channel image (mode `L`, or a two-dimensional array) shows in shades of
  gray, from 0 (black) to 255 (white): without `vmin` and `vmax`, matplotlib would
  stretch each image's values from its minimum to its maximum, and a dull image
  would look contrasted.
  """
  fig, axes = plt.subplots(1, len(images), dpi=dpi, squeeze=False)
  for i, (ax, image) in enumerate(zip(axes[0], images)):
    array = np.asarray(image)
    if array.ndim == 2:
      ax.imshow(array, cmap="gray", vmin=0, vmax=255)
    else:
      ax.imshow(array)
    ax.axis("off")
    if subtitles:
      ax.set_title(subtitles[i])
  if title:
    fig.suptitle(title, y=0.85)
  fig.subplots_adjust(wspace=0, hspace=0)
  plt.show()

# Pillow

## Loading an Image

With Pillow's `Image.open` function, load into an `im` variable an image of your choice from the dataset, in the `lfw` folder (for example `lfw/Chris_Rock/Chris_Rock_0002.jpg`). Keep its path in a `path` variable: the next exercise saves an image next to it.

Display the image's size and its mode (how its colors are encoded), then the image itself with matplotlib's `imshow` function. The following exercises all start from this image `im`.

**Useful documentation**

- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html

In [ ]:
# Your code here

### Additional hints

- `pathlib.Path("lfw/Chris_Rock/Chris_Rock_0002.jpg")` creates a path, which `Image.open` accepts.
- `Image.open` only reads the pixels on first access: in a `with`, call `im.load()` before the end of the block, to use the image after it.
- `im.size` is the pair `(width, height)` and `im.mode` the mode: all LFW photos are 250 × 250 pixels, in `RGB`.
- `plt.axis("off")` removes the axes' ticks.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
path = pathlib.Path(...)
with Image.open(path) as im:
  im.load()
print(f"The image's size is {...}.")
print(f"The image is encoded in {...}.")
plt.imshow(...)
plt.axis("off")
plt.show()
```

### Solution

In [ ]:
path = pathlib.Path("lfw/Chris_Rock/Chris_Rock_0002.jpg")
with Image.open(path) as im:
  # Reads the pixels before the file is closed: the image stays usable after.
  im.load()
print(f"The image's size is {im.size}.")
print(f"The image is encoded in {im.mode}.")
plt.imshow(im)
plt.axis("off")
plt.show()

## Converting to Grayscale and Saving

From your image `im`:

- convert it to grayscale (the `L` mode) with the `convert` method, into an `im_gray` variable;
- save `im_gray` as PNG in the **same folder** as the original photo, under the **same name prefixed with `gray_`** (`gray_Chris_Rock_0002.png`), with the `save` method;
- display `im_gray` next to `im`, and check that the file is in the folder.

**Useful documentation**

- https://pillow.readthedocs.io/en/stable/handbook/concepts.html#modes
- https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.with_name
- https://docs.python.org/3/library/pathlib.html#pathlib.PurePath.with_suffix

In [ ]:
# Your code here

### Additional hints

- `path.name` is the file's name (`Chris_Rock_0002.jpg`), `path.with_name(name)` the same path with another name, and `.with_suffix(".png")` changes its extension.
- `save` infers the format from the `.png` extension.
- `side_by_side((im_gray, im), subtitles=("Grayscale", "Original"))` displays both images side by side.
- In a cell, `!ls {gray_path.parent}` lists the folder: you should see `gray_Chris_Rock_0002.png` in it.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
im_gray = im.convert(...)
gray_path = path.with_name(...).with_suffix(...)
im_gray.save(gray_path)
side_by_side((im_gray, im), subtitles=("Grayscale", "Original"))
```

### Solution

In [ ]:
im_gray = im.convert("L")
gray_path = path.with_name("gray_" + path.name).with_suffix(".png")
# The format, PNG, is inferred from the extension.
im_gray.save(gray_path)
side_by_side((im_gray, im), subtitles=("Grayscale", "Original"))

In [ ]:
!ls {gray_path.parent}

## Geometric Transformations

Apply and display the following transformations of the image `im`:

- a quarter turn clockwise;
- its mirror image (left becomes right);
- an enlargement that doubles its width and height, once with the `NEAREST` interpolation (nearest neighbor) and once with `BICUBIC`: compare the edges of both, the letters in the background for example;
- a reduction that divides its width and height by 5.

Also display the size of the resized images.

**Useful documentation**

- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.transpose
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Transpose
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.rotate

In [ ]:
# Your code here

### Additional hints

- `im.transpose(Image.Transpose.ROTATE_270)` turns the image a quarter turn clockwise (`ROTATE_90`: counterclockwise), and `im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)` flips it like a mirror.
- `im.rotate(-90)` also turns clockwise, but keeps the original size: on an image that isn't square, the corners leave the frame and black bands appear, unless with `expand=True`.
- `im.resize((width, height), Image.Resampling.NEAREST)`: the size is a pair of integers, and `im.width` and `im.height` give those of `im`. With `NEAREST`, each pixel becomes a square of 2 × 2 identical pixels; `BICUBIC` computes each new pixel from its neighbors, and smooths the edges.
- `side_by_side(..., dpi=300)` displays the images larger.
- You should get images of 500 × 500 and 50 × 50 pixels.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
quarter_turn = im.transpose(...)
mirror = im.transpose(...)
side_by_side(
  (quarter_turn, mirror, im), subtitles=("Quarter turn", "Mirror", "Original")
)

enlarged_nearest = im.resize((..., ...), Image.Resampling.NEAREST)
enlarged_bicubic = im.resize((..., ...), Image.Resampling.BICUBIC)
reduced = im.resize(...)
print(enlarged_nearest.size, reduced.size)
side_by_side(
  (enlarged_nearest, enlarged_bicubic), subtitles=("NEAREST", "BICUBIC"), dpi=300
)
side_by_side((reduced, im), subtitles=("Reduced", "Original"))
```

### Solution

In [ ]:
# An exact quarter turn: rotate(-90) would keep the original size, and crop
# an image that isn't square.
quarter_turn = im.transpose(Image.Transpose.ROTATE_270)
mirror = im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
side_by_side(
  (quarter_turn, mirror, im), subtitles=("Quarter turn", "Mirror", "Original")
)

enlarged_nearest = im.resize((im.width * 2, im.height * 2), Image.Resampling.NEAREST)
enlarged_bicubic = im.resize((im.width * 2, im.height * 2), Image.Resampling.BICUBIC)
reduced = im.resize((im.width // 5, im.height // 5))
print(f"Enlarged: {enlarged_nearest.size}, reduced: {reduced.size}")
# NEAREST copies each pixel into a square: staircase edges. BICUBIC smooths
# them.
side_by_side(
  (enlarged_nearest, enlarged_bicubic), subtitles=("NEAREST", "BICUBIC"), dpi=300
)
side_by_side((reduced, im), subtitles=("Reduced", "Original"))

## Filters

A convolution filter replaces each pixel with a weighted sum of its neighbors: these weights make up the filter's kernel. `ImageFilter.Kernel((3, 3), weights)` creates a filter from the kernel's size and its 9 weights, row by row, and `im.filter(image_filter)` applies it.

1. Apply the kernel `[0, -1, 0, -1, 5, -1, 0, -1, 0]` to `im`: each pixel becomes five times its value, minus those of its four direct neighbors. Display the result next to that of the ready-made filter `ImageFilter.SHARPEN` and to the original. What does this kernel do? Which of the two acts more strongly?
2. The Gaussian blur, `ImageFilter.GaussianBlur(radius)`, replaces each pixel with an average of its neighbors, weighted less the farther they are: the radius, in pixels, sets the extent of this neighborhood. Apply it to `im` with a radius of 1, 3 and 8, and display the three results next to the original. Which one erases the face's details (the eyes, the mouth)?

**Useful documentation**

- https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html#PIL.ImageFilter.Kernel
- https://pillow.readthedocs.io/en/stable/reference/ImageFilter.html#PIL.ImageFilter.GaussianBlur

In [ ]:
# Your code here

### Additional hints

- Without a third argument, `ImageFilter.Kernel` divides the weighted sum by the sum of the weights, here 1: in a flat area, a pixel keeps its value; on an edge, its gap with its neighbors grows.
- `ImageFilter.SHARPEN` is a ready-made filter, without parentheses; `ImageFilter.SHARPEN.filterargs` shows its kernel.
- `ImageFilter.GaussianBlur(3)` is a class, instantiated with its radius in pixels.
- `side_by_side(..., dpi=300)` displays the images larger, to see the edges better.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
kernel = ImageFilter.Kernel((3, 3), [...])
sharpened = im.filter(...)
sharpened_pillow = im.filter(...)
side_by_side(
  (sharpened, sharpened_pillow, im),
  subtitles=("Kernel", "SHARPEN", "Original"),
  dpi=300,
)

blurred = [im.filter(ImageFilter.GaussianBlur(...)) for radius in (1, 3, 8)]
side_by_side((*blurred, im), subtitles=("Radius 1", "Radius 3", "Radius 8", "Original"))
```

### Solution

In [ ]:
# Five times the pixel, minus its four direct neighbors: the weights sum to 1, a
# flat area keeps its value and an edge is enhanced.
kernel = ImageFilter.Kernel((3, 3), [0, -1, 0, -1, 5, -1, 0, -1, 0])
sharpened = im.filter(kernel)
sharpened_pillow = im.filter(ImageFilter.SHARPEN)
side_by_side(
  (sharpened, sharpened_pillow, im),
  subtitles=("Kernel", "SHARPEN", "Original"),
  dpi=300,
)
# Both enhance the edges (the hair, the letters in the background). SHARPEN's
# kernel, 2 at the center and -1/8 on the eight neighbors, acts more gently than
# ours.

blurred = [im.filter(ImageFilter.GaussianBlur(radius)) for radius in (1, 3, 8)]
side_by_side((*blurred, im), subtitles=("Radius 1", "Radius 3", "Radius 8", "Original"))
# At radius 1, the face stays sharp; at radius 3, the eyes and the mouth get
# blurred; at radius 8, they are erased, and the face is only a blob.

## Cropping and Pasting

- Crop a rectangle around the face out of `im`, with the `crop` method.
- Create a black image of the same size as `im` with `Image.new`, and paste the cropped face into it, where it was, with the `paste` method.
- Display the collage next to the original.

**Useful documentation**

- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.crop
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.new
- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.Image.paste

In [ ]:
# Your code here

### Additional hints

- `crop` takes a box `(left, top, right, bottom)` in pixels, the origin being at the top left: for `Chris_Rock_0002.jpg`, `(75, 10, 190, 200)` surrounds the face, and gives an image of 115 × 190 pixels.
- `Image.new("RGB", im.size)` creates a black image of the size of `im`.
- `collage.paste(face, (left, top))` pastes `face` with its top left corner at `(left, top)`: it changes `collage` in place, and returns `None`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
face_box = (..., ..., ..., ...)  # (left, top, right, bottom)
face = im.crop(face_box)
collage = Image.new(...)
collage.paste(face, ...)
side_by_side((collage, im), "Cropping and pasting", ("Collage", "Original"))
```

### Solution

In [ ]:
face_box = (75, 10, 190, 200)  # (left, top, right, bottom)
face = im.crop(face_box)

# A black image of the original's size, where the face goes back to its place:
# its top left corner at (left, top).
collage = Image.new("RGB", im.size)
collage.paste(face, face_box[:2])

side_by_side((collage, im), "Cropping and pasting", ("Collage", "Original"))

## Isolating Color Channels

Pillow's `split` method separates an image's channels: each one becomes an `Image` in `L` mode (grayscale).

- Separate the red, green and blue channels of `im`.
- Display each of them in grayscale. Can you see differences?
- Merge the channels back with `Image.merge`, swapping red and blue, and display the result.
- Display each channel again, the red one in red rather than in gray, and likewise for green and blue.

**Useful documentation**

- https://pillow.readthedocs.io/en/stable/reference/Image.html#PIL.Image.merge

In [ ]:
# Your code here

### Additional hints

- `r, g, b = im.split()`; displayed in grayscale, a channel is light where its color is strong.
- `Image.merge("RGB", (b, g, r))` rebuilds an image from three channels, in the given order.
- To see the red channel in red, merge it with two black channels: `black = Image.new("L", im.size)`, then `Image.merge("RGB", (r, black, black))`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
r, g, b = im.split()
side_by_side((r, g, b), subtitles=("red", "green", "blue"))

swapped_channels = Image.merge("RGB", ...)
side_by_side((swapped_channels, im), subtitles=("Red and blue swapped", "Original"))

black = Image.new("L", im.size)
red = Image.merge("RGB", (r, black, black))
green = ...
blue = ...
side_by_side((red, green, blue), subtitles=("red", "green", "blue"), dpi=300)
```

### Solution

In [ ]:
r, g, b = im.split()
# The red carpet, at the bottom left, is light in the red channel and dark in
# the other two.
side_by_side((r, g, b), subtitles=("red", "green", "blue"))

In [ ]:
swapped_channels = Image.merge("RGB", (b, g, r))
side_by_side((swapped_channels, im), subtitles=("Red and blue swapped", "Original"))

In [ ]:
black = Image.new("L", im.size)
red = Image.merge("RGB", (r, black, black))
green = Image.merge("RGB", (black, g, black))
blue = Image.merge("RGB", (black, black, b))
side_by_side((red, green, blue), subtitles=("red", "green", "blue"), dpi=300)

# Image Manipulation With NumPy (*prerequisites: NumPy and matplotlib*)

While Pillow offers many image manipulation methods, the library mostly sticks to classic operations and to loading, saving and mode-changing functions. It lacks advanced operations and flexibility.

NumPy, a scientific computing library, lets you write more complex operations by treating images as multidimensional arrays.

## Redoing Pillow's Transformations With NumPy

A Pillow image converts to a NumPy array of shape `(height, width, channels)` with `np.asarray`: the exercise cell creates this array, `im_np`, and displays its shape and type. The transformations of the Pillow part then become operations on this array: arithmetic and slices.

Redo with NumPy, starting from `im_np` only, four transformations you did with Pillow, and compare each one to Pillow's result:

- grayscale: Pillow computes each gray pixel as the weighted sum `0.299 * red + 0.587 * green + 0.114 * blue` of its three channels, a matrix product; compare your image to `im_gray` by the maximum gap between the two;
- the negative, where each value $v$ becomes $255 - v$; compare it to `ImageOps.invert(im)`;
- the mirror image; compare it to `mirror`;
- the crop around the face, with slices, using the box `face_box` of the cropping exercise; compare it to `face`.

`np.array_equal` tells whether two arrays are identical. Also display your four images.

Success criterion: the maximum gap of the grayscale images is about half a level (0.495 for `Chris_Rock_0002.jpg`), Pillow's rounding to the integer, and the other three comparisons are `True`.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.matmul.html
- https://numpy.org/doc/stable/reference/generated/numpy.array_equal.html
- https://pillow.readthedocs.io/en/stable/reference/ImageOps.html#PIL.ImageOps.invert

In [ ]:
im_np = np.asarray(im)
print(f"Shape: {im_np.shape}, type: {im_np.dtype}")

# Your code here

### Additional hints

- `im_np @ [0.299, 0.587, 0.114]` multiplies the last axis of `im_np`, that of the three channels, by the three weights: an array of shape `(250, 250)`, of floating-point numbers.
- `np.abs(gray_np - np.asarray(im_gray)).max()` gives the maximum gap between two arrays.
- `255 - im_np` stays in `uint8`: each result is between 0 and 255, nothing overflows.
- `im_np[:, ::-1]` goes through the columns backwards.
- `im_np[top:bottom, left:right]` crops, rows first, whereas Pillow's box, `(left, top, right, bottom)`, gives the columns first.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
gray_np = im_np @ [...]
gap = np.abs(gray_np - np.asarray(im_gray)).max()
print(f"Maximum gap of the grayscale images: {gap:.3f}")

negative_np = ...
print(
  "Identical negative:", np.array_equal(negative_np, np.asarray(ImageOps.invert(im)))
)

mirror_np = ...
print("Identical mirror:", np.array_equal(mirror_np, np.asarray(mirror)))

left, top, right, bottom = face_box
face_np = ...
print("Identical face:", np.array_equal(face_np, np.asarray(face)))

side_by_side(
  (gray_np, negative_np, mirror_np, face_np),
  subtitles=("Grayscale", "Negative", "Mirror", "Face"),
)
```

### Solution

In [ ]:
# The matrix product weights the three channels (the last axis) of each pixel.
gray_np = im_np @ [0.299, 0.587, 0.114]
# Pillow rounds each pixel to the integer: a gap of about half a level.
gap = np.abs(gray_np - np.asarray(im_gray)).max()
print(f"Maximum gap of the grayscale images: {gap:.3f}")

negative_np = 255 - im_np
print(
  "Identical negative:", np.array_equal(negative_np, np.asarray(ImageOps.invert(im)))
)

# The columns backwards.
mirror_np = im_np[:, ::-1]
print("Identical mirror:", np.array_equal(mirror_np, np.asarray(mirror)))

# Rows first (top to bottom), then columns (left to right).
left, top, right, bottom = face_box
face_np = im_np[top:bottom, left:right]
print("Identical face:", np.array_equal(face_np, np.asarray(face)))

side_by_side(
  (gray_np, negative_np, mirror_np, face_np),
  subtitles=("Grayscale", "Negative", "Mirror", "Face"),
)

## Swapping and Isolating Channels With NumPy

The last axis of `im_np` is that of the channels, in the order red, green, blue. Rebuild with NumPy, starting from `im_np` only, two images of the channel isolation exercise with Pillow:

- `swapped_channels`, where red and blue are swapped: a single slice, on the last axis, is enough;
- `red`, the red channel alone, the other two at 0: start from an array of zeros of the shape and type of `im_np`, created by `np.zeros_like`, and copy the red channel into it with an assignment.

Check with `np.array_equal` that each one is identical to Pillow's image, and display them.

Success criterion: both comparisons are `True`.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.zeros_like.html

In [ ]:
# Your code here

### Additional hints

- `im_np[:, :, k]` keeps all rows, all columns and only channel `k` (0 for red, 1 for green, 2 for blue).
- `im_np[:, :, ::-1]` goes through the channels backwards: blue, green, red.
- `np.zeros_like(im_np)` creates an array of zeros of the same shape and type (`uint8`), which can be modified, unlike `im_np`: an array created by `np.asarray` from a Pillow image is read-only, and an assignment to it raises `ValueError: assignment destination is read-only`.
- `red_np[:, :, 0] = im_np[:, :, 0]` copies the red channel into the first channel of `red_np`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
swapped_channels_np = im_np[...]
identical = np.array_equal(swapped_channels_np, np.asarray(swapped_channels))
print("Identical swapped channels:", identical)

red_np = np.zeros_like(...)
red_np[...] = ...
print("Identical red channel:", np.array_equal(red_np, np.asarray(red)))

side_by_side(
  (swapped_channels_np, red_np), subtitles=("Red and blue swapped", "Red only")
)
```

### Solution

In [ ]:
# The channels backwards: blue, green, red.
swapped_channels_np = im_np[:, :, ::-1]
identical = np.array_equal(swapped_channels_np, np.asarray(swapped_channels))
print("Identical swapped channels:", identical)

# im_np, created by np.asarray, is read-only: start from an array of zeros, which
# can be modified, and copy the red channel into it.
red_np = np.zeros_like(im_np)
red_np[:, :, 0] = im_np[:, :, 0]
print("Identical red channel:", np.array_equal(red_np, np.asarray(red)))

side_by_side(
  (swapped_channels_np, red_np), subtitles=("Red and blue swapped", "Red only")
)

## Image Histogram

A classic tool in image processing is the [histogram of pixel values](https://en.wikipedia.org/wiki/Image_histogram).

1. Convert your image to grayscale, and keep the associated NumPy array in `im_gray_np`.
2. Write the function `plot_histogram(image)`, which plots the histogram of a grayscale image:
   - on the $x$ axis, the possible values of a pixel (0 to 255);
   - on the $y$ axis, the number of pixels of each value;
   - on the same plot, the curve of the histogram's cumulative sum: at $k$, the number of pixels of value lower than or equal to $k$.

   For the histogram, two ways: count the values yourself with `np.histogram` and plot the bars with `plt.bar`, or let `plt.hist` count.
3. Plot the histogram of `im_gray_np`.

The next exercise uses `plot_histogram`.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.histogram.html
- https://numpy.org/doc/stable/reference/generated/numpy.cumsum.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.bar.html
- https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html

In [ ]:
def plot_histogram(image: np.ndarray) -> None:
  # Your code here
  ...

### Additional hints

- `np.array(im.convert("L"))` gives the grayscale array, of shape `(250, 250)`.
- `counts, edges = np.histogram(image, bins=256, range=(0, 256))` counts the pixels of each value: with `range=(0, 256)`, the 256 bins have a width of 1, and `counts[k]` is the number of pixels of value `k`. Without `range`, the bins only cover the interval between the image's minimum and maximum, and `counts[k]` no longer matches the value `k`.
- `plt.hist(image.ravel(), bins=256, range=(0, 256))` counts and plots at once; `ravel()` flattens the array.
- `cumulative = np.cumsum(counts)` gives the cumulative sum. Its last element is the number of pixels, much larger than the bars: multiply it by `counts.max() / cumulative[-1]` to plot it at the same scale, with `plt.plot(np.arange(256), ...)`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def plot_histogram(image: np.ndarray) -> None:
  counts, _ = np.histogram(image, bins=256, range=(0, 256))
  cumulative = ...
  plt.hist(..., bins=256, range=(0, 256))
  plt.plot(np.arange(256), ...)
  plt.show()


im_gray_np = np.array(...)
plot_histogram(im_gray_np)
```

### Solution

In [ ]:
def plot_histogram(image: np.ndarray) -> None:
  # range=(0, 256): counts[k] is the number of pixels of value k.
  counts, _ = np.histogram(image, bins=256, range=(0, 256))
  cumulative = np.cumsum(counts)
  plt.hist(image.ravel(), bins=256, range=(0, 256))
  # The cumulative sum, brought down to the height of the tallest bar so as not
  # to squash the histogram.
  plt.plot(np.arange(256), cumulative * counts.max() / cumulative[-1])
  plt.xlabel("Pixel value")
  plt.ylabel("Number of pixels")
  plt.show()


im_gray_np = np.array(im.convert("L"))
plot_histogram(im_gray_np)

### Histogram of a Color Image

A color image has one histogram per channel, overlaid here; the scikit-image lab has them computed by `exposure.histogram`.

In [ ]:
for channel, color, name in [(0, "r", "red"), (1, "g", "green"), (2, "b", "blue")]:
  plt.hist(
    im_np[:, :, channel].ravel(),
    bins=256,
    range=(0, 256),
    color=color,
    alpha=0.5,
    label=name,
  )
plt.legend()
plt.show()

## Correcting Contrast With Histogram Equalization

A simple use of the histogram is contrast correction. Depending on the image, the histogram can concentrate on blacks, grays or whites, without covering the whole range of possible values. To try it on a truly dull image, the exercise cell creates `dull`: your grayscale image, brought down to values from 80 to 165 at most.

Equalization replaces each pixel value $k$ with:

$$
T(k) = \frac{255}{n}\sum_{j=0}^k n_j
$$

with:

- $k$ the value of a pixel (between 0 and 255);
- $n_j$ the number of pixels of value $j$;
- $n$ the number of pixels;
- $\sum_{j=0}^k n_j$ the cumulative frequency of pixel values from 0 to $k$.

$T(k)$ only depends on $k$: rather than computing it pixel by pixel, compute it once for each of the 256 possible values, in a `table` array (`table[k]` is $T(k)$), then apply it to the whole image at once by advanced indexing: `table[image]` replaces each pixel of value `k` with `table[k]`.

1. Write the function `equalization(image)`, which returns the equalized image.
2. Apply it to `dull`, and display the result next to `dull`.
3. Plot the histogram of `dull`, then that of the result, with `plot_histogram`.

Success criterion: the equalized image's values go from almost 0 to 255 (from 1 to 255 for `Chris_Rock_0002.jpg`: display its minimum and maximum), and its cumulative curve, which rose all at once for `dull`, is almost a straight line.

**Useful documentation**

- https://numpy.org/doc/stable/user/basics.indexing.html#integer-array-indexing

In [ ]:
# Your grayscale image, brought down to values from 80 to 165 at most.
dull = 80 + im_gray_np // 3
print(f"Values of dull: from {dull.min()} to {dull.max()}")


def equalization(image: np.ndarray) -> np.ndarray:
  # Your code here
  ...

### Additional hints

- The counts $n_j$: `np.histogram(image, bins=256, range=(0, 256))`, with `range` (see the previous exercise's hints).
- `cumulative = np.cumsum(counts)`: `cumulative[k]` is $\sum_{j=0}^k n_j$, and `cumulative[-1]` is $n$.
- `(255 * cumulative / cumulative[-1]).astype(np.uint8)` computes the 256 values of $T$ at once.
- `table[image]` has the shape of `image`: each integer of `image` serves as an index into `table`.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def equalization(image: np.ndarray) -> np.ndarray:
  counts, _ = np.histogram(...)
  cumulative = ...
  table = ...
  return table[image]


equalized = equalization(dull)
print(equalized.min(), equalized.max())
side_by_side((dull, equalized), subtitles=("Dull", "Equalized"))
plot_histogram(dull)
plot_histogram(equalized)
```

### Solution

In [ ]:
def equalization(image: np.ndarray) -> np.ndarray:
  """Return the grayscale image `image`, its histogram equalized."""
  counts, _ = np.histogram(image, bins=256, range=(0, 256))
  cumulative = np.cumsum(counts)
  # table[k] is T(k), for each of the 256 possible values of a pixel.
  table = (255 * cumulative / cumulative[-1]).astype(np.uint8)
  # Advanced indexing: each pixel of value k becomes table[k].
  return table[image]


equalized = equalization(dull)
print(f"Values of the equalized image: from {equalized.min()} to {equalized.max()}")
side_by_side(
  (dull, equalized, im_gray_np), subtitles=("Dull", "Equalized", "Grayscale")
)
plot_histogram(dull)
plot_histogram(equalized)

## Image Blending (difficulty +)

Blend two images into each other, progressively from left to right: on the left, only the first one shows; on the right, only the second one; in the middle, both at 50%. Column $j$ of the result mixes columns $j$ of both images, $a_j$ and $b_j$:

$$
\alpha_j \cdot a_j + (1 - \alpha_j) \cdot b_j
$$

where $\alpha_j$ decreases evenly from 1, for the first column, to 0, for the last one.

1. Load two images of your choice into NumPy arrays `im_1` and `im_2`. They must have the same size: all LFW photos are 250 × 250 pixels.
2. Write the function `blend(im_1, im_2)`, without a loop: store the $\alpha_j$ in an array, and let NumPy's broadcasting apply each one to its whole column, across all rows and channels.
3. Display the result between the two images.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.linspace.html
- https://numpy.org/doc/stable/user/basics.broadcasting.html

In [ ]:
# Your code here

### Additional hints

- `np.linspace(1, 0, width)` returns `width` evenly spaced values, from 1 to 0: the $\alpha_j$, in an array of shape `(width,)`.
- An image has the shape `(height, width, 3)`: `alpha[None, :, None]`, of shape `(1, width, 1)`, broadcasts across all rows and channels, and `alpha * im_1` multiplies each column $j$ by $\alpha_j$.
- The result is an array of floating-point numbers: `.astype(np.uint8)` brings it back to integers from 0 to 255.
- If broadcasting resists you, first write a loop over the columns, then replace it: `result = np.zeros(im_1.shape)`, then for each `j`, `result[:, j] = alpha[j] * im_1[:, j] + (1 - alpha[j]) * im_2[:, j]`.
- Check: the result's first column is that of `im_1`, and its last one that of `im_2` (`np.array_equal(result[:, 0], im_1[:, 0])`).

A skeleton to copy into the exercise cell, replacing each `...`:

```python
def blend(im_1: np.ndarray, im_2: np.ndarray) -> np.ndarray:
  width = im_1.shape[1]
  alpha = np.linspace(...)[None, :, None]
  return (...).astype(np.uint8)


im_1 = np.array(Image.open(...))
im_2 = np.array(Image.open(...))
side_by_side((im_1, blend(im_1, im_2), im_2), subtitles=("im_1", "Blend", "im_2"))
```

### Solution

In [ ]:
def blend(im_1: np.ndarray, im_2: np.ndarray) -> np.ndarray:
  width = im_1.shape[1]
  # alpha goes from 1, on the left, to 0, on the right; its shape (1, width, 1)
  # broadcasts it across all rows and channels.
  alpha = np.linspace(1, 0, width)[None, :, None]
  return (alpha * im_1 + (1 - alpha) * im_2).astype(np.uint8)


im_1 = np.array(Image.open("lfw/Chris_Rock/Chris_Rock_0001.jpg"))
im_2 = np.array(Image.open("lfw/Chris_Rock/Chris_Rock_0002.jpg"))
side_by_side((im_1, blend(im_1, im_2), im_2), subtitles=("im_1", "Blend", "im_2"))

## Color Isolation (difficulty ++)

Let's now put all your skills into practice, with Pillow and NumPy.

You'll find at the root of your folder a file `bandana.jpg`, the image below:
![bandana](https://github.com/shuuchuu/datasets/raw/refs/heads/main/rosie-the-riveter/rosie-the-riveter.jpg)

*Photo: Alfred T. Palmer, 1943, public domain.*

The goal of this exercise is to produce a grayscale image that keeps the red of the bandana.

Here's the information you'll need:

- An `RGB` image whose three channels are identical is a grayscale image.
- The `RGB` mode is poorly suited to picking a color: prefer the [`HSV`](https://en.wikipedia.org/wiki/HSL_and_HSV) mode. Its first channel, `H` (the hue), gives the pixel's color on a color wheel. Pillow expresses it from 0 to 255: 0 is red, and so is 255 (albeit very subtly more purplish). Cyan is therefore around 128, opposite (don't trust the values shown on the drawing below).
  ![](https://upload.wikimedia.org/wikipedia/commons/5/5c/Color_wheel_with_degree.png)
- The second channel, `S` (the saturation), separates a vivid color from a dull color of the same hue: without it, the skin, whose hue is close to red, keeps some of its color.
- A Pillow image turns into a NumPy array with `np.array`, and an array into a Pillow image with `Image.fromarray`.

Success criterion: the bandana keeps all its red, and the rest of the image turns grayscale, apart from the nails and lips, which are red too.

**Useful documentation**

- https://numpy.org/doc/stable/reference/generated/numpy.where.html

In [ ]:
# Your code here

### Additional hints

- `np.array(Image.fromarray(bandana).convert("HSV"))` gives an array of shape `(height, width, 3)`: `[:, :, 0]` is the hue, `[:, :, 1]` the saturation.
- The hue wheel goes through 0: red is both near 255 and near 0. A single interval, `(hue > 235) & (hue < 251)`, misses half of it, and the bandana comes out with gray patches: you need the union of two intervals, `(hue > 235) | (hue < 12)`.
- `& (saturation > 90)` leaves out the poorly saturated tones.
- The resulting mask is a boolean array of shape `(height, width)`: `mask[:, :, None]` adds a channel axis to it, so that it broadcasts across the image's three channels, and likewise for the grayscale image.
- `np.where(mask, image, gray)` takes `image` where `mask` is true, and `gray` elsewhere.

A skeleton to copy into the exercise cell, replacing each `...`:

```python
bandana = np.array(Image.open("bandana.jpg"))
hsv = np.array(Image.fromarray(bandana).convert(...))
hue, saturation = hsv[:, :, 0], hsv[:, :, 1]
red_mask = (...) & (saturation > 90)
gray = np.array(Image.fromarray(bandana).convert(...))
result = np.where(red_mask[:, :, None], bandana, gray[:, :, None])
side_by_side((bandana, result))
```

### Solution

In [ ]:
def isolate_color(image: np.ndarray, mask: np.ndarray) -> np.ndarray:
  """Keep the color of the mask's pixels, and turn the others to grayscale."""
  gray = np.array(Image.fromarray(image).convert("L"))
  # A channel axis, of length 1, which NumPy broadcasts across the image's three
  # channels: (height, width) becomes (height, width, 1).
  return np.where(mask[:, :, None], image, gray[:, :, None])


bandana = np.array(Image.open("bandana.jpg"))
# HSV, to pick a color by its hue.
hsv = np.array(Image.fromarray(bandana).convert("HSV"))
hue, saturation = hsv[:, :, 0], hsv[:, :, 1]
# Red is at both ends of the hue wheel, near 255 and near 0; the saturation
# leaves out the skin, of a close but duller hue.
red_mask = ((hue > 235) | (hue < 12)) & (saturation > 90)
side_by_side((bandana, isolate_color(bandana, red_mask)))